In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [2]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from lightgbm import LGBMClassifier


In [3]:
df = pd.read_csv("transactions.csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset Shape: (1000000, 23)

Columns:
['transaction_id', 'account_id', 'timestamp', 'hour_of_day', 'day_of_week', 'is_weekend', 'amount', 'merchant_category', 'mcc_code', 'merchant_country', 'card_present', 'device_type', 'device_known', 'ip_risk_score', 'is_foreign_txn', 'time_since_last_s', 'velocity_1h', 'amount_vs_avg_ratio', 'account_age_days', 'has_2fa', 'credit_limit', 'is_fraud', 'fraud_pattern']


In [4]:
drop_cols = [
    "transaction_id",
    "account_id",
    "timestamp"
]

df = df.drop(columns=drop_cols, errors="ignore")

In [5]:
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]


TRAIN / TEST SPLIT

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)



Training data: (800000, 19)
Testing data: (200000, 19)


In [7]:
categorical_cols = [
    "merchant_category",
    "merchant_country",
    "device_type"
]

numerical_cols = [
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "amount",
    "mcc_code",
    "card_present",
    "device_known",
    "ip_risk_score",
    "is_foreign_txn",
    "time_since_last_s",
    "velocity_1h",
    "amount_vs_avg_ratio",
    "account_age_days",
    "has_2fa",
    "credit_limit"
]

ENCODE CATEGORICAL FEATURES

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_cols
        ),
        (
            "num",
            "passthrough",
            numerical_cols
        )
    ]
)

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("\nProcessed X_train:", X_train_processed.shape)
print("Processed X_test:", X_test_processed.shape)


Processed X_train: (800000, 45)
Processed X_test: (200000, 45)


LIGHTGBM MODEL

In [10]:
lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,          
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

TRAIN MODEL

In [11]:
lgbm_model.fit(
    X_train_processed,
    y_train
)


,learning_rate,0.05
,n_estimators,300
,subsample,0.8
,colsample_bytree,0.8
,random_state,42
,n_jobs,-1
,is_unbalance,True
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1


In [12]:
y_pred = lgbm_model.predict(
    X_test_processed
)

y_prob = lgbm_model.predict_proba(
    X_test_processed
)[:, 1]


In [13]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\nAccuracy:", accuracy)


Accuracy: 0.965865


In [14]:
print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred
    )
)


Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98    196571
           1       0.33      0.96      0.49      3429

    accuracy                           0.97    200000
   macro avg       0.66      0.96      0.74    200000
weighted avg       0.99      0.97      0.97    200000



In [15]:

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)



Confusion Matrix:
[[189881   6690]
 [   137   3292]]


In [16]:
roc_auc = roc_auc_score(
    y_test,
    y_prob
)

print("\nROC-AUC Score:", roc_auc)



ROC-AUC Score: 0.9947136629219845


In [17]:
print("\nFirst 10 Predictions:")
print(y_pred[:10])

print("\nFirst 10 Fraud Probabilities:")
print(y_prob[:10])


First 10 Predictions:
[0 0 0 0 0 0 0 0 0 0]

First 10 Fraud Probabilities:
[1.32579395e-05 2.85874889e-05 8.92173833e-06 2.66376163e-01
 2.46108089e-05 1.13400496e-05 5.06506222e-06 3.94785778e-06
 4.13630694e-06 1.94514239e-05]


In [18]:
import joblib

joblib.dump(lgbm_model, "fraud_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

print("Models saved successfully!")

Models saved successfully!
